# ViTPose-base + RT-DETR — DIMER two-stage human pose estimation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/vitpose-keypoint-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/vitpose-keypoint-pipeline/blob/main/tutorials/vitpose_keypoint_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-usyd--community%2Fvitpose--base-ffcc4d?style=flat)](https://huggingface.co/usyd-community/vitpose-base) [![Upstream](https://img.shields.io/badge/Upstream-ViTAE--Transformer%2FViTPose-181717?style=flat&logo=github&logoColor=white)](https://github.com/ViTAE-Transformer/ViTPose) [![arXiv](https://img.shields.io/badge/arXiv-2204.12484-b31b1b.svg)](https://arxiv.org/abs/2204.12484)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** two-stage human pose estimation — `person` boxes from the pinned `PekingU/rtdetr_r50vd` detector (or boxes you supply), then 17 COCO keypoints per person from the pinned `usyd-community/vitpose-base` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/vitpose_keypoint_pipeline/pipeline.py` at revision `ebe84a24dcdf`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `95be2991424e646950d656bb7fc15ec9be119700` (~532 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

ViTPose is a **top-down** pose estimator: it needs a person box first. Stage 1 runs the RT-DETR R50-VD detector (the same checkpoint the sibling `rtdetr-detection-pipeline` wraps) over the whole image and keeps the boxes labelled `person` above a threshold; stage 2 affine-warps each box to a 192×256 crop, runs a plain ViT-B backbone (86M parameters) with a heatmap decoder, and reads 17 COCO keypoints — nose, eyes, ears, shoulders, elbows, wrists, hips, knees, ankles — back in input-pixel coordinates, each with the maximum of its heatmap as its score. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the two upstream checkpoints supply the weights and processor configurations, and the carried module adds snapshot verification for both, the input contract (image side ceilings, optional caller boxes, two thresholds), a fixed output contract and the `keypoint_pck`, `validate_inputs` and `evaluation_report` helpers. The default sample is a cartoon person drawn in code whose joint positions are known exactly — and on which the photograph-trained detector finds **no** `person` at the default threshold, so the notebook records that finding and hands the drawn box to stage 2 itself; the resulting PCK is demonstration (plumbing) evidence for one drawing, not a pose benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify both immutable upstream model revisions, draw a synthetic person with known joints (or upload your own photograph) and validate it into an input manifest, run stage 1 and read what a closed-set detector does with a drawing, run stage 2 with a caller-supplied box, read heatmap scores and the two caller-owned thresholds correctly, exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with per-person `keypoint_pck` only when reference joints exist and `not-measurable` otherwise, and export machine-readable keypoints plus a skeleton overlay and provenance.

**This notebook does not demonstrate:** bottom-up or multi-person association without boxes (every person needs a box, from the detector or from you), 3-D pose, tracking across frames, hand, face or whole-body keypoints beyond the 17 COCO joints (ViTPose+ and other checkpoints cover those), COCO OKS-based average precision (which needs a labelled keypoint set; only PCK against joints you drew is computed here), the upstream MMPose evaluation stack, or any training. A box that contains no person still yields 17 keypoints, with low heatmap scores as the only signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 0.3 s for stage 1 on the 640×640 drawing and 0.15 s for stage 2 on one person in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the two checkpoints (360 MB ViTPose, 172 MB RT-DETR) are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what a keypoint heatmap is and why its maximum is not a probability; what percentage-of-correct-keypoints measures.
- **Data:** the default sample is a deterministic 640×640 cartoon person drawn in code with Pillow (head, torso, limbs in flat colours on a plain background) from 17 joint coordinates you can read in the code, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar) — a photograph with one or more people fully visible works best — any colour mode, sides between 16 and 4096 px; on BYOD the detector supplies the boxes. Do not upload confidential or restricted data (photographs of identifiable people you have no consent to process) to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `usyd-community/vitpose-base` snapshot (~532 MB in total) at revision `95be2991424e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'vitpose-keypoint-pipeline',
    'repository_revision': 'ebe84a24dcdf1f285b38b3c4c12dc25f322df49c',
    'embedded_module': 'src/vitpose_keypoint_pipeline/pipeline.py',
    'embedded_modules': ['src/vitpose_keypoint_pipeline/pipeline.py'],
    'module_sha256': '1b64bfb4aa1e42bab7cc3b533246dfa7416fe05d0eac14d2aebf76265e921667',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/vitpose_keypoint_pipeline/` @ `ebe84a24dcdf`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/vitpose_keypoint_pipeline/pipeline.py`

In [ ]:
"""Two-stage human pose estimation with the pinned ``usyd-community/vitpose-base`` checkpoint (ViTPose)
and the pinned ``PekingU/rtdetr_r50vd`` person detector (RT-DETR).

Both models load only from digest-verified local snapshots (``weights/<key>/``) or, when explicitly
allowed, from the Hugging Face Hub at their pinned revisions — always with ``trust_remote_code=False``:
the architectures come from the pinned ``transformers`` release, the weights are SafeTensors, and no
model-repository code is executed. ViTPose is top-down: it needs a person box first, either from the
carried detector stage or from the caller.
"""

from __future__ import annotations

import hashlib
import json
import math
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "usyd-community/vitpose-base"
MODEL_REVISION = "95be2991424e646950d656bb7fc15ec9be119700"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "vitpose-base"
_WEIGHTS_ROOT = Path.cwd() / "weights"  # standalone rewrite (build_notebook.py): working-directory-relative
DEFAULT_WEIGHTS_DIR = _WEIGHTS_ROOT / MODEL_KEY
MANIFEST_NAME = "dimer-base-manifest.json"

# Stage 1 (person detection) is a SECOND pinned snapshot with its own manifest: the RT-DETR R50-VD
# checkpoint the sibling rtdetr-detection-pipeline wraps. Only its `person` class is used here.
DETECTOR_MODEL_ID = "PekingU/rtdetr_r50vd"
DETECTOR_REVISION = "df939e661d8c52e80608d1ec566561aabd25a4e7"
DETECTOR_LICENSE = "apache-2.0"
DETECTOR_KEY = "rtdetr-r50vd"
DETECTOR_WEIGHTS_DIR = _WEIGHTS_ROOT / DETECTOR_KEY
DETECTOR_PERSON_LABEL = "person"
# Detection threshold on the detector's per-class sigmoid for `person`: the value the pinned ViTPose
# README's two-stage example passes (threshold=0.3). Uncalibrated; the deployment owns tuning it.
DETECTION_THRESHOLD = 0.3
# Keypoint threshold on ViTPose's per-keypoint heatmap maximum: the README example's value (0.3).
# Keypoints below it are dropped from `keypoints` but still listed under `all_keypoints`.
KEYPOINT_THRESHOLD = 0.3
# The 17 COCO keypoints in the checkpoint's id2label order (config.json) and its skeleton edges.
KEYPOINT_NAMES = (
    "Nose",
    "L_Eye",
    "R_Eye",
    "L_Ear",
    "R_Ear",
    "L_Shoulder",
    "R_Shoulder",
    "L_Elbow",
    "R_Elbow",
    "L_Wrist",
    "R_Wrist",
    "L_Hip",
    "R_Hip",
    "L_Knee",
    "R_Knee",
    "L_Ankle",
    "R_Ankle",
)
SKELETON_EDGES = (
    (15, 13),
    (13, 11),
    (16, 14),
    (14, 12),
    (11, 12),
    (5, 11),
    (6, 12),
    (5, 6),
    (5, 7),
    (6, 8),
    (7, 9),
    (8, 10),
    (1, 2),
    (0, 1),
    (0, 2),
    (1, 3),
    (2, 4),
    (3, 5),
    (4, 6),
)
# Input ceilings. Each person crop is affine-warped to 192x256 (preprocessor_config.json) so the
# pose stage's cost is per person, not per pixel; the detector resizes the whole image to 640x640.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PERSONS = 50
# PCK normaliser for the sanity metric: a keypoint is "correct" when within this fraction of the
# person box's longest side of its reference (a stated convention, not the COCO OKS metric).
PCK_FRACTION = 0.1


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the ViTPose snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def verify_detector_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the RT-DETR person-detector snapshot against its DIMER manifest."""
    root = Path(path) if path is not None else DETECTOR_WEIGHTS_DIR
    return _verify_manifest(root, DETECTOR_MODEL_ID, DETECTOR_REVISION)


def _hub_download(relative_path: str, root: Path, model_id: str, revision: str) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(model_id, relative_path, revision=revision, local_dir=str(root))


def _stage_missing(
    root: Path,
    model_id: str,
    revision: str,
    allow_download: bool,
    downloader: Callable[[str, Path], None] | None,
) -> list[str]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != model_id or manifest.get("revision") != revision:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {model_id}@{revision}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {revision}"
        )
    fetch = downloader or (lambda rel, dst: _hub_download(rel, dst, model_id, revision))
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch ViTPose manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _stage_missing(root, MODEL_ID, MODEL_REVISION, allow_download, downloader)


def stage_missing_detector_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Same as `stage_missing_files` for the RT-DETR person-detector snapshot at DETECTOR_REVISION."""
    root = Path(path) if path is not None else DETECTOR_WEIGHTS_DIR
    return _stage_missing(root, DETECTOR_MODEL_ID, DETECTOR_REVISION, allow_download, downloader)


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(name: str, value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


def _check_boxes(boxes: Any, width: int, height: int) -> list[list[float]]:
    if isinstance(boxes, str) or not isinstance(boxes, Sequence):
        raise TypeError("person_boxes must be a sequence of [x0, y0, x1, y1] boxes")
    if not 1 <= len(boxes) <= MAX_PERSONS:
        raise ValueError(f"person_boxes has {len(boxes)} entries; expected 1..MAX_PERSONS={MAX_PERSONS}")
    checked: list[list[float]] = []
    for index, box in enumerate(boxes):
        if isinstance(box, str) or not isinstance(box, Sequence) or len(box) != 4:
            raise ValueError(f"person_boxes[{index}] must be [x0, y0, x1, y1]")
        try:
            x0, y0, x1, y1 = (float(v) for v in box)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"person_boxes[{index}] must hold numbers") from exc
        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
            raise ValueError(
                f"person_boxes[{index}] {list(box)} is not a non-empty box inside the {width}x{height} image"
            )
        checked.append([x0, y0, x1, y1])
    return checked


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one image as PIL.Image.Image (any mode, converted to RGB) and optionally the person boxes as "
        "pixel xyxy; without boxes the carried RT-DETR stage detects `person` first"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "person_boxes": [1, MAX_PERSONS],
    "detection_threshold": [0.0, 1.0],
    "keypoint_threshold": [0.0, 1.0],
    "keypoints": list(KEYPOINT_NAMES),
    "preprocessing": (
        "stage 1 (when no boxes are given): the whole image resized to 640x640 for RT-DETR, `person` boxes "
        "kept above detection_threshold; stage 2: each person box affine-warped to a 192x256 crop "
        "(ImageNet mean/std) for ViTPose, heatmap maxima mapped back to input pixels"
    ),
    "output": "per person: the box, its source, 17 named keypoints with pixel coordinates and heatmap scores",
}


def _check_inputs(
    image: Any, person_boxes: Any, detection_threshold: Any, keypoint_threshold: Any
) -> tuple[Image.Image, list[list[float]] | None, float, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``estimate`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    boxes = None if person_boxes is None else _check_boxes(person_boxes, rgb.width, rgb.height)
    return (
        rgb,
        boxes,
        _check_threshold("detection_threshold", detection_threshold),
        _check_threshold("keypoint_threshold", keypoint_threshold),
    )


def validate_inputs(
    image: Image.Image,
    *,
    person_boxes: Sequence[Sequence[float]] | None = None,
    detection_threshold: float = DETECTION_THRESHOLD,
    keypoint_threshold: float = KEYPOINT_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``estimate`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, boxes, det_t, kp_t = _check_inputs(image, person_boxes, detection_threshold, keypoint_threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (estimate takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "person_boxes": boxes,
        "box_source": "caller" if boxes is not None else "detector",
        "detection_threshold": det_t,
        "keypoint_threshold": kp_t,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "detector_model_id": DETECTOR_MODEL_ID,
        "detector_revision": DETECTOR_REVISION,
    }


def keypoint_pck(
    predicted: Mapping[str, Sequence[float]],
    reference: Mapping[str, Sequence[float]],
    box: Sequence[float],
    *,
    fraction: float = PCK_FRACTION,
) -> dict[str, Any]:
    """Percentage of correct keypoints: a named reference joint counts as correct when the predicted
    joint of the same name lies within ``fraction`` of the box's longest side; missing joints are misses."""
    if not reference:
        raise ValueError("reference must contain at least one keypoint")
    if len(box) != 4 or box[2] <= box[0] or box[3] <= box[1]:
        raise ValueError("box must be a non-empty [x0, y0, x1, y1]")
    radius = fraction * max(box[2] - box[0], box[3] - box[1])
    errors: dict[str, float | None] = {}
    correct = 0
    for name, (rx, ry) in reference.items():
        if name in predicted:
            px, py = predicted[name][0], predicted[name][1]
            err = math.hypot(px - rx, py - ry)
            errors[name] = err
            correct += err <= radius
        else:
            errors[name] = None
    measured = [e for e in errors.values() if e is not None]
    return {
        "value": correct / len(reference),
        "correct": correct,
        "total": len(reference),
        "radius_px": radius,
        "fraction": fraction,
        "mean_error_px": sum(measured) / len(measured) if measured else None,
        "errors_px": errors,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_keypoints: Sequence[Mapping[str, Sequence[float]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_keypoints`` (one name -> (x, y) mapping per person, in the order of
    ``result['poses']``) the report carries one ``keypoint_pck`` entry per person as sample-sanity
    geometry evidence; without them the verdict is ``not-measurable`` and the report says what
    labelled data would make the task measurable.
    """
    poses = list(result["poses"])
    base = {
        "task": "two-stage human pose estimation: person boxes -> 17 COCO keypoints per person",
        "score_semantics": (
            "each keypoint score is the maximum of its ViTPose heatmap after the crop's affine warp — a "
            "ranking signal per joint, not a calibrated probability that the joint is where it says; the "
            "person score, when the detector stage ran, is RT-DETR's per-class sigmoid for `person`"
        ),
        "box_source": result.get("box_source"),
        "detection_threshold": result.get("detection_threshold", DETECTION_THRESHOLD),
        "keypoint_threshold": result.get("keypoint_threshold", KEYPOINT_THRESHOLD),
        "sample_kind": sample_kind,
        "n_persons": len(poses),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "detector_model_id": DETECTOR_MODEL_ID,
        "detector_revision": DETECTOR_REVISION,
    }
    if not reference_keypoints:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference keypoints were supplied for the evaluated image",
            "needs": (
                "images with COCO-style keypoint annotations from the deployment domain scored with "
                "OKS-based average precision (the COCO keypoint metric) or PCK at a stated normaliser; "
                "no such labelled set ships with this repository"
            ),
        }
    if len(reference_keypoints) != len(poses):
        raise ValueError(f"reference_keypoints has {len(reference_keypoints)} entries for {len(poses)} poses")
    metrics = []
    for index, (pose, reference) in enumerate(zip(poses, reference_keypoints, strict=True)):
        unknown = set(reference) - set(KEYPOINT_NAMES)
        if unknown:
            raise ValueError(f"unknown keypoint names in reference {index}: {sorted(unknown)}")
        predicted = {kp["name"]: (kp["x"], kp["y"]) for kp in pose["all_keypoints"]}
        pck = keypoint_pck(predicted, reference, pose["box"])
        metrics.append(
            {
                "id": "keypoint_pck",
                "person": index,
                "value": pck["value"],
                "correct": pck["correct"],
                "total": pck["total"],
                "radius_px": pck["radius_px"],
                "mean_error_px": pck["mean_error_px"],
                "estimation": "one drawn person on a single image, no dispersion estimate",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} person(s) on one tutorial image whose joints you drew yourself; geometry "
            "sanity evidence, not a pose benchmark"
        ),
        "needs": (
            "a keypoint-labelled image set from the deployment domain (cameras, poses, occlusion, clothing) "
            "for any OKS-AP or PCK claim"
        ),
    }


@dataclass
class VitPoseKeypointPipeline:
    """``_detect(image, threshold)`` -> [{"box", "score"}] of persons; ``_pose(image, boxes)`` ->
    per box a list of 17 {"name", "x", "y", "score"} in KEYPOINT_NAMES order."""

    _detect: Callable[[Image.Image, float], list[dict[str, Any]]]
    _pose: Callable[[Image.Image, list[list[float]]], list[list[dict[str, Any]]]]
    device: str = "cpu"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        detector_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> VitPoseKeypointPipeline:
        roots = {
            "pose": (
                Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR,
                MODEL_ID,
                MODEL_REVISION,
            ),
            "detector": (
                Path(detector_dir) if detector_dir is not None else DETECTOR_WEIGHTS_DIR,
                DETECTOR_MODEL_ID,
                DETECTOR_REVISION,
            ),
        }
        sources: dict[str, tuple[str, dict[str, Any]]] = {}
        for name, (root, model_id, revision) in roots.items():
            if (root / MANIFEST_NAME).is_file():
                _stage_missing(root, model_id, revision, allow_download, None)
                _verify_manifest(root, model_id, revision)
                sources[name] = (str(root), {"local_files_only": True})
            elif allow_download:
                sources[name] = (model_id, {"revision": revision})
            else:
                raise FileNotFoundError(
                    f"no verified {name} snapshot at {root} and allow_download=False; "
                    f"stage {model_id}@{revision} under {root}"
                )
        # Refuse invalid snapshots before importing model libraries.
        import numpy as np
        import torch
        from transformers import AutoImageProcessor, RTDetrForObjectDetection, VitPoseForPoseEstimation

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        det_src, det_kwargs = sources["detector"]
        det_processor = AutoImageProcessor.from_pretrained(det_src, trust_remote_code=False, **det_kwargs)
        detector = (
            RTDetrForObjectDetection.from_pretrained(
                det_src, trust_remote_code=False, use_pretrained_backbone=False, **det_kwargs
            )
            .to(resolved_device)
            .eval()
        )
        det_id2label = {int(k): v for k, v in detector.config.id2label.items()}
        pose_src, pose_kwargs = sources["pose"]
        pose_processor = AutoImageProcessor.from_pretrained(pose_src, trust_remote_code=False, **pose_kwargs)
        pose_model = (
            VitPoseForPoseEstimation.from_pretrained(pose_src, trust_remote_code=False, **pose_kwargs)
            .to(resolved_device)
            .eval()
        )
        pose_id2label = {int(k): v for k, v in pose_model.config.id2label.items()}
        if tuple(pose_id2label[i] for i in range(len(pose_id2label))) != KEYPOINT_NAMES:
            raise RuntimeError("snapshot id2label does not match KEYPOINT_NAMES")

        def detect(image: Image.Image, threshold: float) -> list[dict[str, Any]]:
            inputs = det_processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = detector(**inputs)
            result = det_processor.post_process_object_detection(
                outputs, threshold=threshold, target_sizes=[image.size[::-1]]
            )[0]
            return [
                {"box": [float(v) for v in box.tolist()], "score": float(score)}
                for box, label, score in zip(result["boxes"], result["labels"], result["scores"], strict=True)
                if det_id2label[int(label)] == DETECTOR_PERSON_LABEL
            ]

        def pose(image: Image.Image, boxes: list[list[float]]) -> list[list[dict[str, Any]]]:
            # The ViTPose processor takes boxes as xywh (the README converts xyxy -> xywh the same way).
            xywh = np.array([[b[0], b[1], b[2] - b[0], b[3] - b[1]] for b in boxes], dtype=np.float32)
            inputs = pose_processor(image, boxes=[xywh], return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = pose_model(**inputs)
            # threshold=0.0 keeps every joint; the pipeline applies keypoint_threshold itself so both
            # the kept and the dropped joints can be reported.
            persons = pose_processor.post_process_pose_estimation(outputs, boxes=[xywh], threshold=0.0)[0]
            out: list[list[dict[str, Any]]] = []
            for person in persons:
                joints = [
                    {
                        "name": pose_id2label[int(label)],
                        "x": float(kp[0]),
                        "y": float(kp[1]),
                        "score": float(score),
                    }
                    for kp, label, score in zip(
                        person["keypoints"], person["labels"], person["scores"], strict=True
                    )
                ]
                out.append(joints)
            return out

        return cls(detect, pose, resolved_device)

    def detect_people(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """Stage 1 alone: `person` boxes from the carried RT-DETR, sorted by score."""
        rgb, _boxes, det_t, _kp_t = _check_inputs(image, None, threshold, KEYPOINT_THRESHOLD)
        found = sorted(self._detect(rgb, det_t), key=lambda d: -d["score"])
        for det in found:
            if set(det) != {"box", "score"} or len(det["box"]) != 4:
                raise RuntimeError(f"detector returned a malformed detection: {det!r}")
        return {
            "persons": found[:MAX_PERSONS],
            "n_persons": min(len(found), MAX_PERSONS),
            "threshold": det_t,
            "width": rgb.width,
            "height": rgb.height,
            "detector_model_id": DETECTOR_MODEL_ID,
            "detector_revision": DETECTOR_REVISION,
        }

    def estimate(
        self,
        image: Image.Image,
        *,
        person_boxes: Sequence[Sequence[float]] | None = None,
        detection_threshold: float = DETECTION_THRESHOLD,
        keypoint_threshold: float = KEYPOINT_THRESHOLD,
    ) -> dict[str, Any]:
        """Keypoints for every person: boxes from the caller, or from stage 1 when none are given."""
        rgb, boxes, det_t, kp_t = _check_inputs(image, person_boxes, detection_threshold, keypoint_threshold)
        if boxes is None:
            detected = self.detect_people(rgb, threshold=det_t)["persons"]
            boxes = [d["box"] for d in detected]
            person_scores: list[float | None] = [d["score"] for d in detected]
            source = "detector"
        else:
            person_scores = [None] * len(boxes)
            source = "caller"
        poses: list[dict[str, Any]] = []
        if boxes:
            per_person = self._pose(rgb, boxes)
            if len(per_person) != len(boxes):
                raise RuntimeError(f"pose stage returned {len(per_person)} results for {len(boxes)} boxes")
            for box, score, joints in zip(boxes, person_scores, per_person, strict=True):
                if [j["name"] for j in joints] != list(KEYPOINT_NAMES):
                    raise RuntimeError("pose stage returned keypoints out of KEYPOINT_NAMES order")
                kept = [j for j in joints if j["score"] >= kp_t]
                poses.append(
                    {
                        "box": [float(v) for v in box],
                        "person_score": score,
                        "keypoints": kept,
                        "all_keypoints": joints,
                        "n_keypoints": len(kept),
                        "mean_keypoint_score": sum(j["score"] for j in joints) / len(joints),
                    }
                )
        return {
            "poses": poses,
            "n_persons": len(poses),
            "box_source": source,
            "detection_threshold": det_t,
            "keypoint_threshold": kp_t,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "detector_model_id": DETECTOR_MODEL_ID,
            "detector_revision": DETECTOR_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `95be2991424e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `VitPoseKeypointPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, detector_dir=DETECTOR_WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The package also pins a second snapshot `rtdetr-r50vd` (4 files), carried and verified the same way. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "vitpose-base",
  "modelId": "usyd-community/vitpose-base",
  "revision": "95be2991424e646950d656bb7fc15ec9be119700",
  "files": [
    {
      "path": "README.md",
      "bytes": 11309,
      "sha256": "6e956c405569ad33e9cce408193376a0104bf610e8928bbfa94759603b541578"
    },
    {
      "path": "config.json",
      "bytes": 1799,
      "sha256": "488377867b16542e0c0025a0eab646abfd066e67f115ab3ec100e3af49872787"
    },
    {
      "path": "model.safetensors",
      "bytes": 360007012,
      "sha256": "cd9a4e6cefc33c51ddcc32dd509fc4114b5845256463a69a10ea2dfde402b2f8"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 363,
      "sha256": "9b11cadc98c30b968a70cc1658ce1fbd74b721b0f21402a2ff8bc1dc9d1474a0"
    }
  ],
  "totalBytes": 360020483
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})

DETECTOR_MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "rtdetr-r50vd",
  "modelId": "PekingU/rtdetr_r50vd",
  "revision": "df939e661d8c52e80608d1ec566561aabd25a4e7",
  "files": [
    {
      "path": "README.md",
      "bytes": 9053,
      "sha256": "4a0c10ddd0dbf6a2c6815cfc1613a5d74f3b0e7c8c77f0252212d3e5365e98cb"
    },
    {
      "path": "config.json",
      "bytes": 5113,
      "sha256": "2ed2a305c51eef46715eb755a02b2a266ecfb752936cc9574bb5714601c2742d"
    },
    {
      "path": "model.safetensors",
      "bytes": 172175856,
      "sha256": "5263d5521eff3e356f6cd8a371fd5dfb891725beda5f713674f79669115cdc64"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 841,
      "sha256": "ffb4b9461a1dad746be8f0f9c8330ed7743a1ba5fba4f75c232cd281b3d4c64a"
    }
  ],
  "totalBytes": 172190863
}

if (DETECTOR_MANIFEST['modelId'], DETECTOR_MANIFEST['revision']) != (DETECTOR_MODEL_ID, DETECTOR_REVISION):
    raise RuntimeError('inline rtdetr-r50vd manifest does not name the identity carried by the pipeline module')
DETECTOR_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(DETECTOR_WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(DETECTOR_MANIFEST, handle, indent=2)
fetched_rtdetr_r50vd = stage_missing_detector_files(DETECTOR_WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(DETECTOR_WEIGHTS_DIR), 'fetched': fetched_rtdetr_r50vd})
_extra = verify_detector_snapshot(DETECTOR_WEIGHTS_DIR)
_extra_files = _extra.get('files', []) if isinstance(_extra, dict) else []
print({'verified_files_rtdetr_r50vd': len(_extra_files) if isinstance(_extra_files, list) else _extra_files})
pipe = VitPoseKeypointPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, detector_dir=DETECTOR_WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic person or optional BYOD

The default sample is **synthetic** and carries its own reference: a 640×640 cartoon person — a skin-tone head with eyes and a mouth, a red shirt torso, red sleeves with skin forearms and hands, blue trousers and dark shoes — is drawn with Pillow from a dictionary of 17 joint coordinates in COCO order (`REFERENCE_JOINTS`), the same drawing the repository's smoke run used. Those coordinates are the reference for the `keypoint_pck` sanity check later, and the drawn figure's bounding box `DRAWN_BOX` is the person box stage 2 will receive; they are not a labelled dataset, so nothing here is a pose benchmark, and a flat cartoon is not a photograph. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one image — stage 1 supplies the boxes and no reference joints exist, so the evaluation report will be `not-measurable`.

Both thresholds are **caller-owned request parameters**, not pipeline constants: `detection_threshold` keeps a `person` box whose RT-DETR sigmoid score reaches it, `keypoint_threshold` keeps a joint whose heatmap maximum reaches it (joints below it are still listed under `all_keypoints`). Their package defaults (`DETECTION_THRESHOLD = 0.3`, `KEYPOINT_THRESHOLD = 0.3`) follow the pinned ViTPose README's two-stage example, not a calibration; they are exposed here as form parameters and passed explicitly on every call. Nothing is validated in this cell — the next section hands the image, the boxes and both thresholds to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the thresholds and the reference joints.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
detection_threshold = 0.3  # @param {type:"number"}
keypoint_threshold = 0.3  # @param {type:"number"}

CX = 320
REFERENCE_JOINTS = {
    'Nose': (CX, 130), 'L_Eye': (CX + 12, 118), 'R_Eye': (CX - 12, 118), 'L_Ear': (CX + 30, 125), 'R_Ear': (CX - 30, 125),
    'L_Shoulder': (CX + 60, 200), 'R_Shoulder': (CX - 60, 200), 'L_Elbow': (CX + 95, 290), 'R_Elbow': (CX - 95, 290),
    'L_Wrist': (CX + 120, 380), 'R_Wrist': (CX - 120, 380), 'L_Hip': (CX + 40, 350), 'R_Hip': (CX - 40, 350),
    'L_Knee': (CX + 50, 450), 'R_Knee': (CX - 50, 450), 'L_Ankle': (CX + 55, 545), 'R_Ankle': (CX - 55, 545),
}
DRAWN_BOX = [180.0, 80.0, 460.0, 560.0]


def cartoon_person(width=640, height=640):
    """A flat cartoon person drawn from REFERENCE_JOINTS: head, torso, arms, legs on a plain background."""
    img = Image.new('RGB', (width, height), (225, 232, 240))
    d = ImageDraw.Draw(img)
    d.rectangle([0, 520, width, height], fill=(150, 160, 140))
    skin, shirt, pants = (222, 180, 140), (200, 60, 60), (40, 60, 140)
    j = REFERENCE_JOINTS
    d.ellipse([CX - 42, 88, CX + 42, 172], fill=skin, outline=(120, 80, 60), width=3)
    for eye in ('L_Eye', 'R_Eye'):
        d.ellipse([j[eye][0] - 5, j[eye][1] - 5, j[eye][0] + 5, j[eye][1] + 5], fill=(30, 30, 30))
    d.arc([CX - 18, 135, CX + 18, 160], 10, 170, fill=(120, 40, 40), width=3)
    d.polygon([j['R_Shoulder'], j['L_Shoulder'], (j['L_Hip'][0] + 10, j['L_Hip'][1]), (j['R_Hip'][0] - 10, j['R_Hip'][1])], fill=shirt)
    d.line([(CX, 172), (CX, 200)], fill=skin, width=18)
    for side in ('L', 'R'):
        d.line([j[f'{side}_Shoulder'], j[f'{side}_Elbow'], j[f'{side}_Wrist']], fill=shirt, width=26, joint='curve')
        d.line([j[f'{side}_Elbow'], j[f'{side}_Wrist']], fill=skin, width=22, joint='curve')
        d.ellipse([j[f'{side}_Wrist'][0] - 14, j[f'{side}_Wrist'][1] - 14, j[f'{side}_Wrist'][0] + 14, j[f'{side}_Wrist'][1] + 14], fill=skin)
        d.line([j[f'{side}_Hip'], j[f'{side}_Knee'], j[f'{side}_Ankle']], fill=pants, width=34, joint='curve')
        d.ellipse([j[f'{side}_Ankle'][0] - 22, j[f'{side}_Ankle'][1] - 10, j[f'{side}_Ankle'][0] + 22, j[f'{side}_Ankle'][1] + 14], fill=(40, 40, 40))
    return img


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    reference_joints, drawn_box = None, None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic drawing: no randomness and no text rendering, so the digest is stable across Pillow builds.
    image = cartoon_person()
    reference_joints, drawn_box = {k: (float(x), float(y)) for k, (x, y) in REFERENCE_JOINTS.items()}, DRAWN_BOX
    image_name = 'synthetic_person_640x640.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'detection_threshold': detection_threshold, 'keypoint_threshold': keypoint_threshold, 'drawn_box': drawn_box, 'n_reference_joints': None if reference_joints is None else len(reference_joints)})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `estimate` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, optional person boxes (1..`MAX_PERSONS`, each a non-empty xyxy box inside the image) and both thresholds in `[0, 1]` — and returns an **input manifest** naming the schema (including the 17 keypoint names and both models' preprocessing), the input's observed mode and size, the boxes and their source (`caller` or `detector`), the thresholds, the verdict and **both** model identities. The manifest is written to `outputs/vitpose_keypoint_input_manifest.json`. To show what rejection looks like, the cell also validates a box that leaves the image and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB; stage 1 resizes it to 640×640 for the detector and stage 2 warps each box to 192×256 for ViTPose; keypoints are mapped back to input pixels, and nothing else is dropped or altered. On the synthetic path the manifest names the drawn box as a caller box, which is what stage 2 will receive.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PERSONS': MAX_PERSONS, 'DETECTION_THRESHOLD': DETECTION_THRESHOLD, 'KEYPOINT_THRESHOLD': KEYPOINT_THRESHOLD, 'KEYPOINT_NAMES': list(KEYPOINT_NAMES), 'PCK_FRACTION': PCK_FRACTION}})
input_manifest = validate_inputs(image, person_boxes=None if drawn_box is None else [drawn_box], detection_threshold=detection_threshold, keypoint_threshold=keypoint_threshold, names=[image_name])
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(image, person_boxes=[[0, 0, image.width + 1, 10]])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'box-outside-image-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/vitpose_keypoint_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in input_manifest.items() if k != 'schema'}, indent=2))

## 6. Stage 1 (detect people), stage 2 (keypoints) — and read the scores correctly

`detect_people` runs stage 1 alone and returns the `person` boxes above `detection_threshold`, sorted by score. On the cartoon the photograph-trained RT-DETR finds **no** person at the default 0.3 — as recorded in the model card, the smoke run got zero boxes at 0.3 and seven low-confidence `person` proposals at 0.05 — so the notebook records that finding and calls `estimate` with the drawn box as a caller box (`box_source: caller`); on BYOD it calls `estimate` without boxes and stage 1 supplies them (`box_source: detector`). `estimate` returns one entry per person in `poses`: the `box`, the detector's `person_score` (or `None` for a caller box), `all_keypoints` — all 17 joints in `KEYPOINT_NAMES` order with pixel `x`, `y` and a `score` — and `keypoints`, the subset at or above `keypoint_threshold`. Each keypoint `score` is the **maximum of that joint's heatmap after the crop's affine warp — a ranking signal per joint, not a calibrated probability**, and a box that contains no person still yields 17 joints (the smoke run's blank box scored every joint below 0.04, which is the only signal you get). Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); CUDA kernel selection can move coordinates by a pixel and scores in the third decimal. The smoke run's stage 2 on this drawing returned all 17 joints at scores 0.85–0.97 with a mean error of about 13 px against the drawn joints; that is one observation on a cartoon, not a calibration point.

In [ ]:
import time

t0 = time.time()
stage1 = pipe.detect_people(image, threshold=detection_threshold)
t_stage1 = time.time() - t0
print({'stage1_persons': stage1['n_persons'], 'stage1_seconds': round(t_stage1, 2), 'top_person_scores': [round(p['score'], 3) for p in stage1['persons'][:3]]})
findings = []
if drawn_box is not None:
    if stage1['n_persons'] == 0:
        findings.append('stage 1 found no `person` on the drawing at the default threshold; stage 2 receives the drawn box as a caller box')
    t0 = time.time()
    result = pipe.estimate(image, person_boxes=[drawn_box], detection_threshold=detection_threshold, keypoint_threshold=keypoint_threshold)
else:
    t0 = time.time()
    result = pipe.estimate(image, detection_threshold=detection_threshold, keypoint_threshold=keypoint_threshold)
t_stage2 = time.time() - t0
print({'box_source': result['box_source'], 'n_persons': result['n_persons'], 'stage2_seconds': round(t_stage2, 2), 'device': pipe.device, 'findings': findings})
for index, pose in enumerate(result['poses']):
    print(f"person {index}: box {[round(v, 1) for v in pose['box']]}  person_score {pose['person_score']}  kept {pose['n_keypoints']}/17  mean score {pose['mean_keypoint_score']:.3f}")
    for kp in pose['all_keypoints']:
        flag = '' if kp['score'] >= keypoint_threshold else '  (below keypoint_threshold)'
        print(f"    {kp['name']:12} x {kp['x']:7.1f}  y {kp['y']:7.1f}  score {kp['score']:.3f}{flag}")
if result['n_persons'] == 0:
    print('No person box was available, so stage 2 did not run; on BYOD, lower detection_threshold or supply boxes.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No pose metric is reported by default: COCO's OKS-based average precision needs a keypoint-labelled image set, and this repository ships none. The repository's metric helper is `keypoint_pck` — percentage of correct keypoints, a joint counting as correct when the predicted joint of the same name lies within `PCK_FRACTION` (0.1) of the person box's longest side of its reference (a stated convention, not OKS) — with the mean pixel error; when reference joints are supplied (one mapping per person, matched to `poses` in order) the report carries one `keypoint_pck` entry per person with the verdict `sample-sanity`, scoring **all 17 predicted joints** regardless of `keypoint_threshold`. On the synthetic path those references are joints **you drew yourself** and the box came from you, so a high PCK proves only that the input contract, the affine crop, the forward pass and the coordinate mapping round-trip. On BYOD no reference exists, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/vitpose_keypoint_evaluation_report.json`.

In [ ]:
references = None if reference_joints is None else [reference_joints] * result['n_persons']
report = evaluation_report(result, references or None, sample_kind=sample_kind)
report['findings'] = findings
with open('outputs/vitpose_keypoint_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k != 'metrics'}, indent=2))
for metric in report['metrics']:
    print(f"person {metric['person']}: keypoint_pck {metric['value']:.3f}  ({metric['correct']}/{metric['total']} within {metric['radius_px']:.0f} px; mean error {metric['mean_error_px']:.1f} px)")
if report['verdict'] == 'not-measurable':
    print('No reference joints exist for this input, so keypoint_pck is not computed; inspect the skeleton overlay instead.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (per person: box, source, all 17 keypoints with scores, the kept subset, both thresholds), the stage-1 detections, the evaluation report, the input manifest, the sample identity, digest, drawn box and reference joints, the notebook's source (repository, revision, embedded module digest, generator), **both** model identifiers and immutable revisions, the model licences, and the runtime identity (Python, `torch`, `transformers`, device). The keypoints are also written as CSV with explicit `image`, `person`, `keypoint`, `x`, `y`, `score`, `kept` columns, and a skeleton overlay PNG draws each person's box, the kept joints and the COCO skeleton edges between kept joints (a supplement to, not a replacement for, the machine-readable files). No credentials are recorded.

In [ ]:
import csv

annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for pose in result['poses']:
    draw.rectangle(pose['box'], outline=(0, 160, 0), width=2)
    kept = {kp['name']: (kp['x'], kp['y']) for kp in pose['keypoints']}
    for a, b in SKELETON_EDGES:
        na, nb = KEYPOINT_NAMES[a], KEYPOINT_NAMES[b]
        if na in kept and nb in kept:
            draw.line([kept[na], kept[nb]], fill=(255, 200, 0), width=3)
    for x, y in kept.values():
        draw.ellipse([x - 4, y - 4, x + 4, y + 4], fill=(220, 30, 30))
annotated.save('outputs/vitpose_keypoint_annotated.png')
payload = {
    'prediction': result,
    'stage1': stage1,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'drawn_box': drawn_box, 'reference_joints': reference_joints},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'detector_model_id': DETECTOR_MODEL_ID,
    'detector_revision': DETECTOR_REVISION,
    'detector_license': DETECTOR_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/vitpose_keypoint_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/vitpose_keypoint_keypoints.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'person', 'keypoint', 'x', 'y', 'score', 'kept'])
    for index, pose in enumerate(result['poses']):
        for kp in pose['all_keypoints']:
            writer.writerow([image_name, index, kp['name'], f"{kp['x']:.2f}", f"{kp['y']:.2f}", f"{kp['score']:.6f}", kp['score'] >= keypoint_threshold])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The keypoints are where ViTPose's heatmaps peak inside a box it was told contains a person; the score is a heatmap maximum, not a calibrated probability, and both thresholds are request parameters you own (the defaults are the upstream README example's values, not tuned operating points). On the synthetic drawing the `keypoint_pck` value in the evaluation report compares joints to a figure you drew yourself, inside a box you supplied, and the verdict is `sample-sanity`, which proves only that the input contract, the crop, the forward pass and the coordinate mapping work; it says nothing about photographs, occlusion, crowds, unusual poses, children, clothing, or camera viewpoints, and a BYOD result is a single-image observation with the verdict `not-measurable`. **Stage 1 is a closed-set photograph detector**: it found no `person` on this cartoon at the default threshold, so on drawings, sketches or stylised imagery you must supply boxes yourself, and on photographs a missed or merged person silently loses a pose. **Stage 2 returns 17 joints for any box** — an empty box in the smoke run scored every joint below 0.04 — so a box without a person produces a low-scoring skeleton rather than nothing, and only the scores tell you. The pipeline provides no bottom-up association, no 3-D pose, no tracking, no hand or face keypoints, no OKS evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify both pinned models, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** lower `detection_threshold` to 0.05 and count the low-confidence `person` proposals stage 1 now returns on the drawing (the smoke run found seven); pass a box that contains only background to `estimate` and read the scores; move an elbow in `REFERENCE_JOINTS`, redraw, and watch the PCK error for that joint; enable `USE_BYOD` with a photograph of a person, let stage 1 supply the box, then hand-annotate a few joints and pass them as reference joints to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/vitpose-keypoint-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/vitpose-keypoint-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/vitpose-keypoint-pipeline/blob/main/docs/WEIGHTS.md
- Upstream pose model: https://huggingface.co/usyd-community/vitpose-base
- Upstream person detector: https://huggingface.co/PekingU/rtdetr_r50vd
- Upstream code: https://github.com/ViTAE-Transformer/ViTPose
- ViTPose: Simple Vision Transformer Baselines for Human Pose Estimation (Xu et al., 2022): https://arxiv.org/abs/2204.12484
- DETRs Beat YOLOs on Real-time Object Detection (Zhao et al., 2023): https://arxiv.org/abs/2304.08069
- Microsoft COCO: Common Objects in Context — keypoint annotations (Lin et al., 2014): https://arxiv.org/abs/1405.0312